# 04 – Data Transformation: Bronze → Silver

Cleans and standardizes bronze data:
- Column renaming (PEP 8 snake_case)
- Type casting (year, age, rating to proper types)
- Data quality filters (valid ranges for year, age, rating)
- Location parsing (city, state, country from compound string)
- Deduplication

**Source:** `bronze.{books_raw, users_raw, ratings_raw}`
**Target:** `silver.{books, users, ratings}`
**Config:** `config/04_data_transformation.json`

## Transformation Summary

### Applied to All Three Tables

| Transformation | Purpose |
| --- | --- |
| Column rename | Maps bronze names (`Book_Title`, `User_ID`, …) to PEP 8 `snake_case` for cleaner downstream SQL and joins |
| `try_cast` to int | CSV stores all values as strings; `try_cast` converts valid values and returns NULL for garbage, avoiding hard failures |
| Null primary-key filter | Rows without a valid PK (`isbn`, `user_id`) are unusable in joins |

### Books-Specific

| Transformation | Purpose |
| --- | --- |
| Year range filter (1900–2025) | Drops rows with nonsensical years (0, 2050, publisher names that leaked into the year column) |
| Deduplication on `isbn` | A few ISBNs appear more than once in the raw file |
| Trim `title`, `author`, `publisher` | Removes leading/trailing whitespace that causes false mismatches in joins and aggregations |

### Users-Specific

| Transformation | Purpose |
| --- | --- |
| Age range filter (5–120) | Ages outside this range are set to NULL (not dropped) — 39.7 % of users have no age, so dropping would lose too much data |
| Location parsing | Splits the compound string `"city, state, country"` into three separate columns (`city`, `state`, `country`) for richer analytics |
| Deduplication on `user_id` | PK hygiene |

### Ratings-Specific

| Transformation | Purpose |
| --- | --- |
| Rating range filter (0–10) | 0 = implicit interaction, 1–10 = explicit rating; anything outside is invalid |
| Null `isbn` + `user_id` filter | Both are foreign keys; rows missing either cannot be joined downstream |

All thresholds (`min_year`, `max_age`, `min_rating`, …) are read from `config/04_data_transformation.json` — nothing is hardcoded in the source code.

In [0]:
import json, sys, importlib

SETUP_CONFIG = "/Workspace/Users/tharcisio.leone@x1f-fink.one/poc/config/00_setup.json"
with open(SETUP_CONFIG) as f:
    setup = json.load(f)

sys.path.insert(0, setup["project_path"])

for mod in [m for m in list(sys.modules.keys()) if m.startswith("source")]:
    del sys.modules[mod]

DataTransformation = importlib.import_module("source.04_data_transformation").DataTransformation

CONFIG_PATH = f"{setup['project_path']}/config/04_data_transformation.json"

In [0]:
pipeline = DataTransformation(config_path=CONFIG_PATH)
dataframes = pipeline.run()

In [0]:
display(dataframes["books"])

In [0]:
display(dataframes["users"])

In [0]:
display(dataframes["ratings"])